In [ ]:
using PyPlot
using LinearAlgebra
using SparseArrays

# The linear and nonlinear (elastic) wave equation

The linear wave equation:
$$
\begin{cases}
  \partial_t\,u \,+\,\partial_x\,v\,&=\,0,
  \\[0.3em]
  \partial_t\,v \,+c^2\,\partial_x\,u\,&=\,0.
\end{cases}
$$

The nonlinear elasticity-wave equation:
$$
\begin{cases}
  \partial_t\,u \,+\,\partial_x\,v\,&=\,0,
  \\[0.3em]
  \partial_t\,v \,+c^2\,(1+\partial_x\,u)(1+\tfrac12\partial_x\,u)\partial_x\,u\,&=\,0.
\end{cases}
$$

# Problem definition

In [ ]:
# Gaussian centered in domain:

c = 1.0
stab = 1.0

@inline function analytic_solution(t, x)
    u = 0.25 * exp(-4 * (x-2)^2)
    v = 0
    return u, v
end

@inline function f(t, x)
    return 0, 0
end

;

In [ ]:
# Manufactured solution consisting of a traveling wave to the right.

c = 0.25
stab = 1.0

factor = 1.0 / π^2

@inline function analytic_solution(t, x)
    u = factor * sin(π * (x - c * t))
    v = factor * c * sin(π * (x - c * t))
    return u, v
end

@inline function f(t, x)
    u_dx = factor * π * cos(π * (x - c * t))
    v_dt = -factor * π * c^2 * cos(π * (x - c * t))
    return 0, v_dt + c^2 * (1 + u_dx) * (1 + 0.5 * u_dx) * u_dx
end

;

# Finite difference scheme with stabilization:

In [ ]:
@enum BoundaryType do_nothing reflecting dirichlet periodic

struct Discretization
    xs::Vector{Float64}
    left_boundary::BoundaryType
    right_boundary::BoundaryType
end

In [ ]:
# Captures c, analytic_solution, f

function euler_step_linear(discretization, CFL, stabilization, U_old, t_old)
    h = discretization.xs[2] - discretization.xs[1]

    u_old = U_old[1,:]
    v_old = U_old[2,:]

    τ = CFL * h

    u_new = zeros(2^k + 2)
    v_new = zeros(2^k + 2)

    #
    # Central difference for interior degrees of freedom:
    #
    
    for i in 2 : 2^k + 1
        #f_i = f(t, discretization.xs[i])
        
        u_new[i] = u_old[i] - τ / (2.0 * h) * (v_old[i+1] - v_old[i-1]) #+ τ * f_i[1]
        u_prime = 1.0 / (2.0 * h) * (u_old[i+1] - u_old[i-1])
        v_new[i] = v_old[i] - c^2 * τ * u_prime #+ τ * f_i[2]

        u_new[i] = u_new[i] + stabilization * τ * (u_old[i+1] - 2.0 * u_old[i] + u_old[i-1])
        v_new[i] = v_new[i] + stabilization * τ * (v_old[i+1] - 2.0 * v_old[i] + v_old[i-1])
    end

    #
    # One-sided difference for boundary degrees of freedom:
    #

    f_1 = f(t, discretization.xs[1])
    u_new[1] = u_old[1] - τ / (1.0 * h) * (v_old[2] - v_old[1]) + τ * f_1[1]
    u_prime = 1.0 / (1.0 * h) * (u_old[2] - u_old[1])
    v_new[1] = v_old[1] - c^2 * τ * u_prime + τ * f_1[2]

    N = 2^k + 2
    f_N = f(t, discretization.xs[N])
    u_new[N] = u_old[N] - τ / (1.0 * h) * (v_old[N] - v_old[N-1]) + τ * f_N[2]
    u_prime = 1.0 / (1.0 * h) * (u_old[N] - u_old[N-1])
    v_new[N] = v_old[N] - c^2 * τ * u_prime + τ * f_N[2]

    #
    # Boundary post-processing:
    #

    if discretization.left_boundary == periodic || discretization.right_boundary ==  periodic
        @assert(discretization.left_boundary == periodic && discretization.right_boundary == periodic)
        
        f_1 = f(t, discretization.xs[1])
        u_new[1] = u_old[1] - τ / (2.0 * h) * (v_old[2] - v_old[N-1]) + τ * f_1[1]
        u_prime = 1.0 / (2.0 * h) * (u_old[2] - u_old[N-1])
        v_new[1] = v_old[1] - c^2 * τ * u_prime + τ * f_1[2]
        u_new[1] = u_new[1] + stabilization * τ * (u_old[2] - 2.0 * u_old[1] + u_old[N-1])
        v_new[1] = v_new[1] + stabilization * τ * (v_old[2] - 2.0 * v_old[1] + v_old[N-1])
        u_new[N] = u_new[1]
        v_new[N] = v_old[1]
    end

    if discretization.left_boundary == reflecting
        v_new[1] = 0.
    elseif discretization.left_boundary == dirichlet
        u_new[1], v_new[1] = analytic_solution(t + τ, discretization.xs[1])
    end

    if discretization.right_boundary == reflecting
        v_new[N] = 0.
    elseif discretization.right_boundary == dirichlet
        u_new[N], v_new[N] = analytic_solution(t + τ, discretization.xs[N])
    end
            
    return vcat(u_new', v_new'), τ
end

In [ ]:
# Captures c, analytic_solution, f

function euler_step_nonlinear(discretization, CFL, stabilization, U_old, t_old) 
    h = discretization.xs[2] - discretization.xs[1]

    u_old = U_old[1,:]
    v_old = U_old[2,:]

    τ = CFL * h

    u_new = zeros(2^k + 2)
    v_new = zeros(2^k + 2)

    #
    # Central difference for interior degrees of freedom:
    #
    
    for i in 2 : 2^k + 1
      f_i = f(t, discretization.xs[i])
        
      u_new[i] = u_old[i] - τ / (2.0 * h) * (v_old[i+1] - v_old[i-1]) + τ * f_i[1]
      u_prime = 1.0 / (2.0 * h) * (u_old[i+1] - u_old[i-1])
      v_new[i] = v_old[i] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime + τ * f_i[2]

      u_new[i] = u_new[i] + stabilization * τ * (u_old[i+1] - 2.0 * u_old[i] + u_old[i-1])
      v_new[i] = v_new[i] + stabilization * τ * (v_old[i+1] - 2.0 * v_old[i] + v_old[i-1])
    end

    #
    # One-sided difference for boundary degrees of freedom:
    #

    f_1 = f(t, discretization.xs[1])
    u_new[1] = u_old[1] - τ / (1.0 * h) * (v_old[2] - v_old[1]) + τ * f_1[1]
    u_prime = 1.0 / (1.0 * h) * (u_old[2] - u_old[1])
    v_new[1] = v_old[1] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime + τ * f_1[2]

    N = 2^k + 2
    f_N = f(t, discretization.xs[N])
    u_new[N] = u_old[N] - τ / (1.0 * h) * (v_old[N] - v_old[N-1]) + τ * f_N[1]
    u_prime = 1.0 / (1.0 * h) * (u_old[N] - u_old[N-1])
    v_new[N] = v_old[N] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime + τ * f_N[2]

    #
    # Boundary value post-processing:
    #

    if discretization.left_boundary == periodic || discretization.right_boundary ==  periodic
        @assert(discretization.left_boundary == periodic && discretization.right_boundary == periodic)
        
        f_1 = f(t, discretization.xs[1])
        u_new[1] = u_old[1] - τ / (2.0 * h) * (v_old[2] - v_old[N-1]) + τ * f_1[1]
        u_prime = 1.0 / (2.0 * h) * (u_old[2] - u_old[N-1])
        v_new[1] = v_old[1] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime + τ * f_1[2]
        u_new[1] = u_new[1] + stabilization * τ * (u_old[2] - 2.0 * u_old[1] + u_old[N-1])
        v_new[1] = v_new[1] + stabilization * τ * (v_old[2] - 2.0 * v_old[1] + v_old[N-1])
        u_new[N] = u_new[1]
        v_new[N] = v_old[1]
    end
    
    if discretization.left_boundary == reflecting
        v_new[1] = 0.
    elseif discretization.left_boundary == dirichlet
        u_new[1], v_new[1] = analytic_solution(t + τ, discretization.xs[1])
    end

    if discretization.right_boundary == reflecting
        v_new[N] = 0.
    elseif discretization.right_boundary == dirichlet
        u_new[N], v_new[N] = analytic_solution(t + τ, discretization.xs[N])
    end
    
    return vcat(u_new', v_new'), τ
end

# The main loop (explicit Euler updates to final time)

In [ ]:
# The domain Ω
Ω = [0.0, 4.0]
left_boundary = dirichlet
right_boundary = dirichlet

# Spatial discretization
k = 9
h = (Ω[2] - Ω[1]) / (2^k + 1)
xs = [Ω[1] + h * (i - 1) for i in 1:(2^k+2)]

discretization = Discretization(xs, left_boundary, right_boundary);

In [ ]:
# Final time
T = 4.
CFL = 0.1
out = 1.0e-5

output_granularity = 4.0
yrange = [-1/π^2,1/π^2] # for plotting

;

In [ ]:
@time begin
    U = zeros(2, 2^k + 2) # The two-component state vector [u, v]
    U[1,:] = [analytic_solution(0, x)[1] for x in xs]
    U[2,:] = [analytic_solution(0, x)[2] for x in xs]
    U_linear = copy(U)

    U_new = copy(U)
    U_linear_new = copy(U)
    
    print("output at initial time t = 0\n")
    plt.figure(figsize=(10,7))
    #plot(xs, U[1,:])
    plot(xs, U_linear[1,:])
    plot(xs, [analytic_solution(0, x)[1] for x in xs ])
    plt.xlim(Ω)
    plt.ylim(yrange)
    plt.savefig("output-" * lpad(0,4,"0") * ".png")
    plt.close()
    output_cycle = 1
    
    t = 0
    while t < T
        U, τ = euler_step_nonlinear(discretization, CFL, stab, U, t)
        U_linear, τ_linear = euler_step_linear(discretization, CFL, stab, U_linear, t)
        
        @assert(abs(τ - τ_linear) < 1.e-10)
        if(τ < out * T)
            print("We didn't make it.\n")
            break
        end
            
        t = t + τ
    
        if (output_cycle * output_granularity < t)
            print("output at time t = ", t, " (τ = ", τ, ")\n")
            plt.figure(figsize=(10,7))
            #plot(xs, U[1,:])
            plot(xs, U_linear[1,:])
            plot(xs, [analytic_solution(t, x)[1] for x in xs ])
            plt.xlim(Ω)
            plt.ylim(yrange)
            plt.savefig("output-" * lpad(output_cycle,4,"0") * ".png")
            plt.close()
            output_cycle = output_cycle + 1
        end
    end;
    
    # Compute error at final time:
    U_error = zeros(2, 2^k + 2) # The two-component state vector [u, v]
    U_error[1,:] = [analytic_solution(t, x)[1] for x in xs]
    U_error[2,:] = [analytic_solution(t, x)[2] for x in xs]
    U_error = U_error - U
    error_nonlinear = norm(U_error[1,:], Inf) + norm(U_error[2,:], Inf)
    println("L^∞ error nonlinear: ", error_nonlinear)
    
    U_error = zeros(2, 2^k + 2) # The two-component state vector [u, v]
    U_error[1,:] = [analytic_solution(t, x)[1] for x in xs]
    U_error[2,:] = [analytic_solution(t, x)[2] for x in xs]
    U_error = U_error - U_linear
    error_linear = norm(U_error[1,:], Inf) + norm(U_error[2,:], Inf)
    println("L^∞ error linear:    ", error_linear)
end